# Chapter 17: Query Rewriting and Expansion

Record original and rewritten queries; compare term coverage, candidate lists, and selection.

In [ ]:
import sys
from pathlib import Path

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
if str(ROOT / "examples") not in sys.path:
    sys.path.insert(0, str(ROOT / "examples"))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)


In [ ]:
from document_identity import Collection, digest_bytes
from evidence_units import evidence_success, version_key
from query_retrieval import Retrieval, interpret, reformulate, Budget
import json
from pathlib import Path

fixture = json.loads((ROOT / "experiments" / "part4-complex" / "development-corpus.json").read_text())
relations = json.loads((ROOT / "experiments" / "part4-complex" / "development-relations.json").read_text())
config = json.loads((ROOT / "experiments" / "part4-complex" / "config.json").read_text())
needs = json.loads((ROOT / "experiments" / "part4-complex" / "development-queries.json").read_text())["queries"]

def engine(view="main"):
    return Retrieval(Collection(fixture), view, relations, config)


In [ ]:
# Reformulation preserves the original query and records changes/uncertainties.
p = engine().plan("purge scratch buffer", "q")
t = reformulate(p, config)
print("original:", repr(t["original_query"]))
print("new:     ", repr(t["new_query"]))
print("changes:", t["changes"])
print("uncertainties:", t["uncertainties"])
assert t["original_query"] == p["original_query"]
assert "temporary cache" in t["new_query"]
assert "scratch buffer" in t["new_query"]


In [ ]:
# Whitespace normalisation does not overwrite the original request.
p = engine().plan("  envelope   checksum ", "q")
t = reformulate(p, config)
print("original:", repr(t["original_query"]))
print("new:     ", repr(t["new_query"]))
assert t["new_query"] == "envelope checksum"
assert t["original_query"] == "  envelope   checksum "


In [ ]:
# Safe acronym expansion is refused with an uncertainty; unsafe control is explicitly labelled.
safe = reformulate(engine().plan("DR window", "q"), config)
unsafe = reformulate(engine().plan("DR window", "q"), config, unsafe=True)
print("safe new:", repr(safe["new_query"]), "uncertainties:", safe["uncertainties"], "changes:", safe["changes"])
print("unsafe new:", repr(unsafe["new_query"]), "change type:", unsafe["changes"][0]["type"])
assert safe["new_query"] == "DR window"
assert safe["changes"] == []
assert "unsafe" in unsafe["changes"][0]["type"]
assert "disaster recovery" in unsafe["new_query"]


In [ ]:
# Expansion can increase discovery without improving the final top-k: compare original vs expanded candidate lists and selection.
q = next(x for x in needs if x["query_id"] == "development-05")
e = engine()
p = e.plan(q["query"], q["query_id"])
rewritten = e.execute(p, "rewritten")
multi = e.execute(p, "multi-query")
print("rewritten candidates:", len(rewritten["candidates"]), "selected:", len(rewritten["selection"]["selected"]), "chars:", rewritten["selection"]["characters"])
print("multi-query candidates:", len(multi["candidates"]), "selected:", len(multi["selection"]["selected"]), "chars:", multi["selection"]["characters"])
print("multi-query origins:", sorted({o['method'] for c in multi['candidates'] for o in c['origins']}))
assert len(multi["candidates"]) >= len(rewritten["candidates"]) or len(multi["candidates"]) >= 1
assert multi["cost"]["calls"] == 2


In [ ]:
# Expansion preserves identifiers; rewriting cannot swap a literal device id.
p = engine().plan("purge scratch buffer DV-41", "q")
t = reformulate(p, config)
print("identifiers in original:", p["interpretation"]["identifiers"])
print("identifiers in rewritten:", __import__("re").findall(r"\b[A-Z]{2}-\d{2}\b", t["new_query"]))
assert "DV-41" in t["new_query"]
assert p["allowed_sources"] == t["allowed_sources"]
assert p["budgets"] == t["budgets"]


## Observed versus not established
Observed: original-query retention, recorded transformations, identifier preservation, and a case where an extra expanded search adds candidates/calls. Not established: that rewriting improves final top-k, answer quality, or generalises beyond the authored glossary.